Day 25 - Performance Segmentation & Comparative Analysis

In [26]:
import pandas as pd
import numpy as np

In [27]:
df = pd.read_csv("Data_Co_Supply_Chain_Dataset.csv" , encoding="latin1")
df.shape

(180519, 31)

In [28]:
df.columns

Index(['Benefit per order', 'Sales per customer', 'Delivery Status',
       'Late_delivery_risk', 'Category Name', 'Customer City',
       'Customer Country', 'Customer Fname', 'Customer Id', 'Customer Lname',
       'Customer Segment', 'Customer State', 'Department Name', 'Market',
       'Order Country', 'order date (DateOrders)', 'Order Id',
       'Order Item Discount', 'Order Item Discount Rate',
       'Order Item Product Price', 'Order Item Profit Ratio',
       'Order Item Quantity', 'Sales', 'Order Item Total',
       'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status',
       'Product Name', 'shipping date (DateOrders)', 'Shipping Mode'],
      dtype='object')

In [29]:
df["order date (DateOrders)"] = pd.to_datetime(
     df["order date (DateOrders)"] , format="mixed"
)
df["shipping date (DateOrders)"] = pd.to_datetime(
     df["shipping date (DateOrders)"] , format="mixed"
)

In [30]:
df["Shipping_Days"] = (
    df["shipping date (DateOrders)"]
    - df["order date (DateOrders)"]
).dt.days

In [31]:
df[[
    "order date (DateOrders)",
    "shipping date (DateOrders)",
    "Shipping_Days"
]].head()

,order date (DateOrders),shipping date (DateOrders),Shipping_Days
0,2018-01-31 22:56:00,2018-02-03 22:56:00,3
1,2018-01-13 12:27:00,2018-01-18 12:27:00,5
2,2018-01-13 12:06:00,2018-01-17 12:06:00,4
3,2018-01-13 11:45:00,2018-01-16 11:45:00,3
4,2018-01-13 11:24:00,2018-01-15 11:24:00,2


### Build the Regional KPI Table

In [32]:
region_kpi = df.groupby("Order Region").agg(
     Total_Sales = ("Sales" , "sum"),
     Total_Profit = ("Order Profit Per Order" , "sum"),
     Total_Orders = ("Order Id" , "nunique"),
     Average_Order_Value = ("Sales" , "mean"),
     Average_Shipping_Days = ("Shipping_Days" , "mean"),
     Late_Risk_Score = ("Late_delivery_risk" , "mean")
)
region_kpi["Late_Risk_Score"] =(
     region_kpi["Late_Risk_Score"]
)*100

In [33]:
# Profit margin 
region_kpi["Profit_Margin_%"] = (
    region_kpi["Total_Profit"]
    .div(region_kpi["Total_Sales"])
    .mul(100)
)
region_kpi

,Total_Sales,Total_Profit,Total_Orders,Average_Order_Value,Average_Shipping_Days,Late_Risk_Score,Profit_Margin_%
Order Region,,,,,,,
Canada,1.868610e+05,23900.709960,309,194.849889,3.320125,48.800834,12.790633
Caribbean,1.651019e+06,171825.640024,2806,198.487537,3.478240,53.077663,10.407246
Central Africa,3.272630e+05,33447.269960,556,195.147893,3.546213,57.960644,10.220302
Central America,5.665712e+06,616341.570651,9396,199.912216,3.486010,54.754596,10.878448
Central Asia,1.098399e+05,13045.280051,184,198.625556,3.394213,55.334539,11.876628
East Africa,3.762349e+05,43167.729927,613,203.150593,3.492441,55.939525,11.473611
East of USA,1.371112e+06,156263.300194,2323,198.280837,3.471728,55.661605,11.396830
Eastern Asia,1.486401e+06,147368.010614,3318,204.176008,3.500275,54.326923,9.914416
Eastern Europe,7.742666e+05,79717.049920,1292,197.516981,3.484439,55.663265,10.295815


### REVENUE VS PROFITABILITY

In [34]:
# Top 10 by sales
top_sales_region = region_kpi.sort_values(
     by="Total_Sales" ,
     ascending=False
).head(10)
top_sales_region[
     ["Total_Sales" , "Total_Profit" , "Profit_Margin_%"]
]

,Total_Sales,Total_Profit,Profit_Margin_%
Order Region,,,
Western Europe,5.894381e+06,625446.080548,10.610887
Central America,5.665712e+06,616341.570651,10.878448
South America,2.960881e+06,335154.400817,11.319413
Northern Europe,2.155831e+06,233450.600647,10.828801
Southern Europe,2.047919e+06,230829.229883,11.271405
Oceania,2.016654e+06,201478.020484,9.990707
Southeast Asia,1.932496e+06,211342.819786,10.936264
Caribbean,1.651019e+06,171825.640024,10.407246
West of USA,1.571416e+06,164940.660455,10.496308


In [35]:
# Top 10 regions by Profit Margin
top_margin_regions = region_kpi.sort_values(
    by="Profit_Margin_%",
    ascending=False
).head(10)

top_margin_regions[
    ["Total_Sales", "Total_Profit", "Profit_Margin_%"]
]

,Total_Sales,Total_Profit,Profit_Margin_%
Order Region,,,
Southern Africa,2.282516e+05,30826.050146,13.505295
Canada,1.868610e+05,23900.709960,12.790633
Central Asia,1.098399e+05,13045.280051,11.876628
East Africa,3.762349e+05,43167.729927,11.473611
East of USA,1.371112e+06,156263.300194,11.396830
US Center,1.151356e+06,131094.229875,11.386075
South America,2.960881e+06,335154.400817,11.319413
Southern Europe,2.047919e+06,230829.229883,11.271405
South of USA,7.857839e+05,88114.879898,11.213627


### Revenue × Margin Segmentation


In [37]:
# Calculate the benchmarks
sales_threshold = region_kpi["Total_Sales"].median()
margin_threshold = region_kpi["Profit_Margin_%"].median()
(sales_threshold , margin_threshold)

(1371111.987017012, 10.878448457271988)

In [44]:
# create the segmentation
conditions = [
     (region_kpi["Total_Sales"] >= sales_threshold) &
     (region_kpi["Profit_Margin_%"] >= margin_threshold) ,

     (region_kpi["Total_Sales"] >= sales_threshold) &
     (region_kpi["Profit_Margin_%"] < margin_threshold),

     (region_kpi["Total_Sales"] < sales_threshold) &
     (region_kpi["Profit_Margin_%"] >= margin_threshold)
]

choices =  [
     "High Revenue / High Margin" ,
     "High Revenue / Low Margin" ,
     "Low Revenue / High Margin"
]
region_kpi["Performance Segment"] = np.select(
     conditions , choices , default="Low Revuene / Low Margin"
)
region_kpi["Performance Segment"].value_counts()

Performance Segment
Low Revenue / High Margin     7
High Revenue / Low Margin     7
High Revenue / High Margin    5
Low Revuene / Low Margin      4
Name: count, dtype: int64

In [45]:
region_kpi[
    [
        "Total_Sales",
        "Profit_Margin_%",
        "Late_Risk_Score",
        "Average_Shipping_Days",
        "Performance Segment"
    ]
].sort_values(
    by="Total_Sales",
    ascending=False
)

,Total_Sales,Profit_Margin_%,Late_Risk_Score,Average_Shipping_Days,Performance Segment
Order Region,,,,,
Western Europe,5.894381e+06,10.610887,55.848611,3.471467,High Revenue / Low Margin
Central America,5.665712e+06,10.878448,54.754596,3.486010,High Revenue / High Margin
South America,2.960881e+06,11.319413,54.308671,3.477603,High Revenue / High Margin
Northern Europe,2.155831e+06,10.828801,54.044118,3.479984,High Revenue / Low Margin
Southern Europe,2.047919e+06,11.271405,54.384477,3.401124,High Revenue / High Margin
Oceania,2.016654e+06,9.990707,54.020497,3.469452,High Revenue / Low Margin
Southeast Asia,1.932496e+06,10.936264,55.529930,3.476780,High Revenue / High Margin
Caribbean,1.651019e+06,10.407246,53.077663,3.478240,High Revenue / Low Margin
West of USA,1.571416e+06,10.496308,53.959715,3.478794,High Revenue / Low Margin


### Analyze High Revenue / Low Margin

In [47]:
high_revenue_low_margin = region_kpi[
    region_kpi["Performance Segment"] == "High Revenue / Low Margin"
]

In [48]:
high_revenue_low_margin[
    [
        "Total_Sales",
        "Total_Profit",
        "Profit_Margin_%",
        "Late_Risk_Score",
        "Average_Shipping_Days"
    ]
].sort_values(
    by="Total_Sales",
    ascending=False
)

,Total_Sales,Total_Profit,Profit_Margin_%,Late_Risk_Score,Average_Shipping_Days
Order Region,,,,,
Western Europe,5.894381e+06,625446.080548,10.610887,55.848611,3.471467
Northern Europe,2.155831e+06,233450.600647,10.828801,54.044118,3.479984
Oceania,2.016654e+06,201478.020484,9.990707,54.020497,3.469452
Caribbean,1.651019e+06,171825.640024,10.407246,53.077663,3.478240
West of USA,1.571416e+06,164940.660455,10.496308,53.959715,3.478794
South Asia,1.553681e+06,165703.900124,10.665247,56.266977,3.476652
Eastern Asia,1.486401e+06,147368.010614,9.914416,54.326923,3.500275


### Compare Operational Risk Within This Segment

In [49]:
high_revenue_low_margin[
    [
        "Profit_Margin_%",
        "Late_Risk_Score",
        "Average_Shipping_Days"
    ]
].mean()

Profit_Margin_%          10.416230
Late_Risk_Score          54.506358
Average_Shipping_Days     3.479266
dtype: float64

In [50]:
high_revenue_low_margin.loc[
    high_revenue_low_margin["Late_Risk_Score"].idxmax()
]

Total_Sales                         1553680.920196
Total_Profit                         165703.900124
Total_Orders                                  3335
Average_Order_Value                     200.967652
Average_Shipping_Days                     3.476652
Late_Risk_Score                          56.266977
Profit_Margin_%                          10.665247
Performance Segment      High Revenue / Low Margin
Name: South Asia, dtype: object

In [51]:
high_revenue_low_margin.loc[
    high_revenue_low_margin["Profit_Margin_%"].idxmin()
]

Total_Sales                         1486401.338174
Total_Profit                         147368.010614
Total_Orders                                  3318
Average_Order_Value                     204.176008
Average_Shipping_Days                     3.500275
Late_Risk_Score                          54.326923
Profit_Margin_%                           9.914416
Performance Segment      High Revenue / Low Margin
Name: Eastern Asia, dtype: object

### High Revenue / High Margin

In [53]:
high_revenue_high_margin = region_kpi[
    region_kpi["Performance Segment"] == "High Revenue / High Margin"
]

In [54]:
high_revenue_high_margin[
    [
        "Total_Sales",
        "Total_Profit",
        "Profit_Margin_%",
        "Late_Risk_Score",
        "Average_Shipping_Days"
    ]
].sort_values(
    by="Total_Sales",
    ascending=False
)

,Total_Sales,Total_Profit,Profit_Margin_%,Late_Risk_Score,Average_Shipping_Days
Order Region,,,,,
Central America,5.665712e+06,616341.570651,10.878448,54.754596,3.486010
South America,2.960881e+06,335154.400817,11.319413,54.308671,3.477603
Southern Europe,2.047919e+06,230829.229883,11.271405,54.384477,3.401124
Southeast Asia,1.932496e+06,211342.819786,10.936264,55.529930,3.476780
East of USA,1.371112e+06,156263.300194,11.396830,55.661605,3.471728


In [55]:
high_revenue_high_margin.loc[
    high_revenue_high_margin["Total_Sales"].idxmax()
]

Total_Sales                          5665712.101056
Total_Profit                          616341.570651
Total_Orders                                   9396
Average_Order_Value                      199.912216
Average_Shipping_Days                       3.48601
Late_Risk_Score                           54.754596
Profit_Margin_%                           10.878448
Performance Segment      High Revenue / High Margin
Name: Central America, dtype: object

In [56]:
high_revenue_high_margin.loc[
    high_revenue_high_margin["Profit_Margin_%"].idxmax()
]

Total_Sales                          1371111.987017
Total_Profit                          156263.300194
Total_Orders                                   2323
Average_Order_Value                      198.280837
Average_Shipping_Days                      3.471728
Late_Risk_Score                           55.661605
Profit_Margin_%                            11.39683
Performance Segment      High Revenue / High Margin
Name: East of USA, dtype: object

### Final Cross-Segment Comparison

In [59]:
segment_summary = region_kpi.groupby(
    "Performance Segment"
).agg(
    Average_Sales=("Total_Sales", "mean"),
    Average_Profit=("Total_Profit", "mean"),
    Average_Margin=("Profit_Margin_%", "mean"),
    Average_Late_Risk=("Late_Risk_Score", "mean"),
    Average_Shipping_Days=("Average_Shipping_Days", "mean"),
    Number_of_Regions=("Total_Sales", "count")
)
segment_summary

,Average_Sales,Average_Profit,Average_Margin,Average_Late_Risk,Average_Shipping_Days,Number_of_Regions
Performance Segment,,,,,,
High Revenue / High Margin,2.795624e+06,309986.264266,11.160472,54.927856,3.462649,5
High Revenue / Low Margin,2.332769e+06,244316.130414,10.416230,54.506358,3.479266,7
Low Revenue / High Margin,5.094683e+05,58597.021413,11.891396,53.893757,3.431696,7
Low Revuene / Low Margin,7.277384e+05,74144.897484,10.202017,55.856244,3.485529,4


In [60]:
segment_summary.sort_values(
    by="Average_Sales",
    ascending=False
)

,Average_Sales,Average_Profit,Average_Margin,Average_Late_Risk,Average_Shipping_Days,Number_of_Regions
Performance Segment,,,,,,
High Revenue / High Margin,2.795624e+06,309986.264266,11.160472,54.927856,3.462649,5
High Revenue / Low Margin,2.332769e+06,244316.130414,10.416230,54.506358,3.479266,7
Low Revuene / Low Margin,7.277384e+05,74144.897484,10.202017,55.856244,3.485529,4
Low Revenue / High Margin,5.094683e+05,58597.021413,11.891396,53.893757,3.431696,7
